# Intake phase pipeline: minimal version

In [ ]:
import os, sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

NEW_CODE = Path.cwd() if (Path.cwd() / "functions.py").exists() else Path.cwd() / "new_code"
sys.path[:0] = [str(NEW_CODE), str(NEW_CODE.parent)]
import functions as f

## Parameters

In [ ]:
# >>> UPDATE: set to where YOUR epoch files live (DATA_ROOT/<subject>/EEG/processed/<file>) <<<
DATA_ROOT = Path("/Volumes/CENSORLAB$/Or/Cortical Communication/CorticalCommunication/Data")
INTAKE_FILENAME = "{subject}_intake_stim-epo.fif"

SUBJECT = "sub_102"   # sub_102-sub_113 (no sub_106)
EPOCH = 0             # epoch shown step by step
DEMO = False or os.environ.get("PHASE_NOTEBOOK_DEMO") == "1"   # True = synthetic epochs, no data needed

BAND = (4.0, 8.0)              # Hz, frequency band whose phase is estimated (theta)
CUTOFF_MS = 0.0                # ms, stimulus time = where phase is read
FILTER_ORDER = 350             # FIR order of the non-causal filter (longer = sharper band edges)
TOLERANCES_DEG = (45, 30, 15)  # +/- deg around 0 (positive) / 180 (negative) counted as a BOSS hit
TOLERANCE_DEG = 30             # tolerance shaded in the histograms

CAUSAL_PARAMS = {              # causal (online) estimator, used when RUN_OPTIMIZATION = False
    "window_ms": 510.0,        # ms of pre-stimulus data it may use
    "filter_order": FILTER_ORDER,
    "edge": 65,                # samples dropped at both window ends (filter transients)
    "ar_order": 78,            # autoregressive model order used to forecast past t=0
    "hilbert_window": 128,     # samples in the Hilbert transform
    "offset": 0,               # phase-index correction, samples
}

RUN_OPTIMIZATION = False       # True = tune the causal params for this subject (about a minute)
OPTIMIZATION_CONFIG = {
    "n_opt_trials": 200,       # parameter sets tried
    "train_fraction": 0.8,     # epochs used for fitting; the rest are held out
    "random_seed": 0,
    "min_usable_trials": 20,
    "window_ms_range": (300, 950), "window_ms_step": 10,
    "filter_order_range": (150, 450), "filter_order_step": 5,
    "edge_range": (20, 200), "edge_step": 5,
    "ar_order_range": (5, 80),
    "hilbert_window": 128, "offset": 0, "infeasible_penalty": 10.0,
}

LOAD_CONFIG = f.LoadConfig(
    data_root=DATA_ROOT, intake_filename=INTAKE_FILENAME,
    channel="Fz_hjorth",                       # Fz minus the mean of its 4 neighbours
    hjorth_channel="Fz_hjorth",
    hjorth_weights={"Fz": 1, "AF3": -0.25, "AF4": -0.25, "FC1": -0.25, "FC2": -0.25},
    hjorth_scale_reference="Fz",
    condition_column="Condition",              # metadata column with the BOSS label
    condition_auto_keywords=("condition", "boss", "classification", "class"),
    lowpass_before_downsample_hz=100.0, downsample=True, downsample_fs=1000.0,
)

## 1. Load: `X` is (n_epochs, n_times)

In [ ]:
data = f.make_synthetic_epochs(subject=SUBJECT, n_epochs=40, seed=0) if DEMO else f.get_data(SUBJECT, LOAD_CONFIG)
print(f"{SUBJECT}: X {data.X.shape}, fs {data.fs:g} Hz")
pd.Series(data.labels).value_counts()

## 2. Causal parameters

With optimization, the non-causal filter uses the optimized `filter_order` too (both methods share one band-pass).

In [ ]:
if RUN_OPTIMIZATION:
    params = f.optimize_causal_params(data.X, data.times_ms, data.fs, BAND, CUTOFF_MS, OPTIMIZATION_CONFIG, name=SUBJECT)
else:
    params = f.validate_causal_params(CAUSAL_PARAMS)
noncausal_filter_order = f.resolve_noncausal_filter_order(params, FILTER_ORDER)
params

## 3. One epoch, step by step

In [ ]:
x = data.X[EPOCH]
nc = f.noncausal_phase(x, data.times_ms, data.fs, BAND, noncausal_filter_order, CUTOFF_MS)
c = f.causal_phase(x, data.times_ms, data.fs, BAND, CUTOFF_MS, params)
f.plot_epoch_estimates(x, data.times_ms, nc, c, BAND, CUTOFF_MS,
                       title=f"{SUBJECT}, epoch {EPOCH} (BOSS {data.labels[EPOCH]})")
plt.show()

## 4. All epochs: BOSS success per tolerance

In [ ]:
estimates = f.estimate_subject_trials(data, BAND, noncausal_filter_order, CUTOFF_MS, n_trials=False, causal_params=params)
analysis = f.analyze_phase_results({SUBJECT: f.estimates_to_arrays(estimates)}, TOLERANCES_DEG)
success = analysis["success"]["by_tolerance"]
pd.DataFrame([{"tolerance": f"+/-{t}", "correct": success[str(t)]["correct"][0], "n": success[str(t)]["n_labeled"][0],
               "success %": round(success[str(t)]["success_pct"][0], 1), "chance %": round(success[str(t)]["chance_pct"], 1)}
              for t in TOLERANCES_DEG]).set_index("tolerance")

## 5. Results

In [ ]:
deviations = analysis["subjects"][SUBJECT]["deviations"]
fig, axes = plt.subplots(1, 2, figsize=(12, 6), subplot_kw={"projection": "polar"})
f.plot_deviation_histogram(deviations["boss_target"], TOLERANCE_DEG, title="Phase - BOSS target", ax=axes[0])
f.plot_deviation_histogram(deviations["causal_error"], TOLERANCE_DEG, title="Causal - non-causal", ax=axes[1])
f.plot_success_vs_tolerance(analysis["success"])
plt.show()